#  Hyperparameter Tuning

This notebook explores different hyperparameter configurations.

In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import itertools

from src.models import Gemma3Model, GEMMA3_CONFIG_270M

%matplotlib inline
sns.set_style("whitegrid")

## 1. Learning Rate Comparison

In [ ]:
# Define learning rate schedules
def create_lr_schedule(iterations, lr_start=1e-4, lr_end=1e-5):
    """Create a cosine decay learning rate schedule."""
    return lr_end + (lr_start - lr_end) * (1 + np.cos(np.pi * np.arange(iterations) / iterations)) / 2

# Compare different schedules
iterations = 10000
schedules = {
    'constant': np.ones(iterations) * 1e-4,
    'cosine': create_lr_schedule(iterations, 1e-4, 1e-5),
    'linear': np.linspace(1e-4, 1e-5, iterations),
    'warmup_cosine': np.concatenate([
        np.linspace(0, 1e-4, 1000),
        create_lr_schedule(iterations-1000, 1e-4, 1e-5)
    ])
}

plt.figure(figsize=(12, 6))
for name, schedule in schedules.items():
    plt.plot(schedule, label=name, alpha=0.7)

plt.xlabel('Iterations')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedules Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

## 2. Batch Size Impact

In [ ]:
# Analyze batch size impact on training
batch_sizes = [4, 8, 16, 32, 64]

# Estimate memory requirements
def estimate_batch_memory(batch_size, seq_length=128):
    hidden_size = 640
    n_layers = 18
    
    # Rough estimate
    param_memory = 270e6 * 2  # 270M params * 2 bytes for bfloat16
    activation_memory = batch_size * seq_length * hidden_size * n_layers * 2 * 4  # 4 bytes per float
    
    return (param_memory + activation_memory) / 1e9  # GB

memory_estimates = [estimate_batch_memory(b) for b in batch_sizes]

plt.figure(figsize=(12, 6))
plt.bar([str(b) for b in batch_sizes], memory_estimates, color='skyblue')
plt.xlabel('Batch Size')
plt.ylabel('Estimated Memory (GB)')
plt.title('Memory Usage vs Batch Size')
plt.grid(True, alpha=0.3)
for i, (bs, mem) in enumerate(zip(batch_sizes, memory_estimates)):
    plt.text(i, mem + 0.1, f'{mem:.2f} GB', ha='center')
plt.show()

## 3. Model Size Variations

In [ ]:
# Define different model configurations
model_configs = {
    'small': {'emb_dim': 256, 'n_heads': 4, 'n_layers': 6, 'hidden_dim': 768},
    'medium': {'emb_dim': 512, 'n_heads': 8, 'n_layers': 12, 'hidden_dim': 1536},
    'large': {'emb_dim': 768, 'n_heads': 12, 'n_layers': 18, 'hidden_dim': 2304},
}

def estimate_params(config):
    vocab_size = 50257
    emb_dim = config['emb_dim']
    n_heads = config['n_heads']
    n_layers = config['n_layers']
    hidden_dim = config['hidden_dim']
    head_dim = emb_dim // n_heads
    
    # Embeddings
    emb_params = vocab_size * emb_dim
    
    # Attention per layer
    attn_params = 3 * emb_dim * head_dim + emb_dim * emb_dim  # Q, K, V projections + output
    
    # Feedforward per layer
    ff_params = 2 * emb_dim * hidden_dim + hidden_dim * emb_dim
    
    # Norms
    norm_params = 4 * emb_dim * n_layers  # 2 norms per layer, scale parameter
    
    # Output
    out_params = emb_dim * vocab_size
    
    total = emb_params + n_layers * (attn_params + ff_params) + norm_params + out_params
    return total

for name, config in model_configs.items():
    params = estimate_params(config)
    print(f"{name:8s}: {params/1e6:.2f}M parameters")

## 4. Optimal Parameter Finder

In [ ]:
# Grid search over hyperparameters
def hyperparameter_grid():
    """Generate hyperparameter grid."""
    learning_rates = [1e-5, 3e-5, 1e-4, 3e-4]
    batch_sizes = [8, 16, 32]
    gradient_accumulation = [8, 16, 32]
    
    combinations = list(itertools.product(learning_rates, batch_sizes, gradient_accumulation))
    
    results = []
    for lr, bs, ga in combinations:
        effective_batch = bs * ga
        if effective_batch <= 512:  # Reasonable limit
            results.append({
                'learning_rate': lr,
                'batch_size': bs,
                'gradient_accumulation': ga,
                'effective_batch': effective_batch
            })
    
    return pd.DataFrame(results)

grid_df = hyperparameter_grid()
print(f"Total configurations: {len(grid_df)}")
grid_df.head(10)

In [ ]:
# Visualize hyperparameter space
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter plot
scatter = axes[0].scatter(grid_df['learning_rate'], grid_df['effective_batch'], 
                         c=grid_df['gradient_accumulation'], cmap='viridis', s=100)
axes[0].set_xlabel('Learning Rate')
axes[0].set_ylabel('Effective Batch Size')
axes[0].set_title('Hyperparameter Space')
axes[0].set_xscale('log')
axes[0].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[0], label='Gradient Accumulation')

# Distribution
axes[1].hist(grid_df['effective_batch'], bins=10, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Effective Batch Size')
axes[1].set_ylabel('Count')
axes[1].set_title('Effective Batch Size Distribution')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Recommended Configurations

In [ ]:
# Recommendations based on hardware
recommendations = {
    'Low Memory (8GB)': {
        'batch_size': 4,
        'gradient_accumulation': 32,
        'effective_batch': 128,
        'learning_rate': 1e-4,
        'block_size': 64,
    },
    'Medium Memory (16GB)': {
        'batch_size': 8,
        'gradient_accumulation': 32,
        'effective_batch': 256,
        'learning_rate': 1e-4,
        'block_size': 128,
    },
    'High Memory (24GB+)': {
        'batch_size': 16,
        'gradient_accumulation': 32,
        'effective_batch': 512,
        'learning_rate': 1e-4,
        'block_size': 256,
    },
    'Large Model (40GB+)': {
        'batch_size': 32,
        'gradient_accumulation': 16,
        'effective_batch': 512,
        'learning_rate': 1e-4,
        'block_size': 512,
    }
}

# Create comparison table
rec_df = pd.DataFrame(recommendations).T
rec_df

In [ ]:
# Save recommendations
rec_df.to_csv('../logs/hyperparameter_recommendations.csv')
print("Recommendations saved to logs/hyperparameter_recommendations.csv")

print("\nRecommended Training Command:")
print("="*60)
print("python scripts/train.py \\")
print("    --learning_rate 1e-4 \\")
print("    --batch_size 8 \\")
print("    --gradient_accumulation 32 \\")
print("    --block_size 128 \\")
print("    --max_iters 150000")